# Practical 12 — Text Similarity & Cosine Similarity

**Course:** NLP
**Name:**  <!-- fill in -->
**Date:**  <!-- fill in -->

## Aim
To compute pairwise cosine similarity between reviews using TF-IDF vectors, find which reviews are most similar to a given query review, and test whether thematically related reviews actually score as more similar to each other.

## Theory

**Cosine similarity** measures how alike two vectors are by the angle between them, not their magnitude: `cos(theta) = (A . B) / (|A| |B|)`, giving a score from 0 (completely unrelated direction) to 1 (identical direction) for non-negative vectors like TF-IDF weights. This matters because two documents of very different lengths can still be considered similar if they use the same words in similar proportions — cosine similarity ignores raw length in a way that a simpler distance metric wouldn't.

Worth noting explicitly, given the last two practicals: **this is a genuinely different kind of technique than Word2Vec (Practical 8) or LDA (Practical 10).** Those are both *statistical models* that need to learn patterns from co-occurrence across many documents — which is exactly why they broke down on our 15-review corpus. Cosine similarity is a direct geometric computation on vectors that already exist (from Practical 7's TF-IDF) — there's nothing being "learned" or estimated from limited data. So if similarity scores come out low here, that's not necessarily a small-data failure the way it was for the last two practicals — it may just honestly reflect that these particular reviews don't share much vocabulary.

## Algorithm

1. Build a TF-IDF matrix for all 15 reviews and compute the full pairwise cosine similarity matrix.
2. Confirm the diagonal (self-similarity) is 1.0, as a sanity check.
3. Query review 4 (the "Robert De Niro" acting-focused review) and find its most similar other reviews - check whether they're thematically related or not.
4. Query review 10 (price/value-focused) and do the same.
5. Compute the average pairwise similarity across the whole corpus, to have a baseline for what "actually similar" looks like versus typical background similarity for this dataset.

In [ ]:
import sys, os
sys.path.append(os.path.abspath("../python"))

import pandas as pd

import vectorizers
import similarity as sim

pd.set_option("display.max_colwidth", None)

df = pd.read_csv("../datasets/sample_reviews.csv")
tfidf_matrix, feature_names = vectorizers.tfidf_vectorize(df["review"].tolist())
sim_matrix = sim.build_similarity_matrix(tfidf_matrix)

print(f"Similarity matrix shape: {sim_matrix.shape}")


### Step 1 — Sanity check: is the diagonal actually 1.0?

In [ ]:
import numpy as np
diagonal_values = np.diag(sim_matrix)
print(f"Diagonal values: {diagonal_values}")
print(f"All equal to 1.0 (within floating point tolerance)? {np.allclose(diagonal_values, 1.0)}")


### Step 2 — Most similar reviews to review 4 (acting-focused)

In [ ]:
query_id = 4
query_index = df[df["id"] == query_id].index[0]

print(f"Query - Review {query_id}: {df.loc[query_index, 'review']}")
print()

top_matches = sim.most_similar_documents(query_index, sim_matrix, top_k=3)
for match_index, score in top_matches:
    match_id = df.loc[match_index, "id"]
    print(f"  Review {match_id} (similarity {score:.4f}): {df.loc[match_index, 'review']}")


### Step 3 — Most similar reviews to review 10 (price/value-focused)

In [ ]:
query_id = 10
query_index = df[df["id"] == query_id].index[0]

print(f"Query - Review {query_id}: {df.loc[query_index, 'review']}")
print()

top_matches = sim.most_similar_documents(query_index, sim_matrix, top_k=3)
for match_index, score in top_matches:
    match_id = df.loc[match_index, "id"]
    print(f"  Review {match_id} (similarity {score:.4f}): {df.loc[match_index, 'review']}")


**For both queries: are the top matches actually about the same theme as the query, or just reviews that happen to share a few common words (like "movie" or "was") without real thematic connection? Look at the actual similarity scores too, not just the ranking - a "top match" with a very low score isn't meaningfully similar, it's just the least-dissimilar option available.**

### Step 4 — Baseline: average similarity across the whole corpus

In [ ]:
avg_similarity = sim.average_pairwise_similarity(sim_matrix)
print(f"Average pairwise similarity across all 15 reviews: {avg_similarity:.4f}")

# Compare this baseline against the actual top-match scores from Steps 2 and 3 above


---
## Output

*Run every cell above top to bottom, then paste or describe your actual output here — the diagonal sanity check, both queries' top-3 matches with their scores, and the average pairwise similarity baseline. Don't fill this in until you've actually run it.*


## Observations & Conclusion

Answer these based on what you actually saw when you ran the notebook:

- Did review 4's most similar matches actually relate to acting/performance, or were they just the least-dissimilar options among 14 largely unrelated reviews? Look at the actual scores, not just the ranking.
- Same question for review 10's price/value matches.
- How do the top-match scores from Steps 2 and 3 compare to the average pairwise similarity baseline from Step 4? Are the "top matches" meaningfully above that baseline, or barely different from it?
- Given everything from Practicals 8 and 10: is low similarity here a sign this technique also needs more data to work properly, or is it a legitimate result given how little vocabulary overlap 15 short, topically varied reviews would realistically have?

*(Write 4-6 sentences here in your own words once you've run the notebook.)*


---
## Viva Prep — Practice Questions

1. **Why is cosine similarity preferred over a raw distance metric (like Euclidean distance) for comparing TF-IDF vectors?**
   Because cosine similarity only considers the angle/direction between vectors, not their magnitude - so two documents can be judged similar based on proportional word usage even if one is much longer than the other, which a magnitude-sensitive distance metric would penalize.

2. **What should the diagonal of a cosine similarity matrix always be, and why?**
   1.0 for every entry, since any vector has an angle of 0 with itself (identical direction), and cos(0) = 1 - this is a basic sanity check that the similarity computation is implemented correctly.

3. **Why doesn't cosine similarity on TF-IDF vectors suffer from the same small-data problem as Word2Vec or LDA?**
   Because it's a direct geometric computation on vectors that already exist from TF-IDF - it isn't estimating any statistical parameters from co-occurrence patterns across many documents the way Word2Vec and LDA do, so there's no "not enough data to learn from" failure mode in the same sense.

4. **If two documents share zero vocabulary, what will their cosine similarity be, and why?**
   0 (or very close to it) - with no shared terms, their TF-IDF vectors have no overlapping non-zero dimensions, so their dot product is 0, making the cosine of the angle between them 0 regardless of either vector's magnitude.

5. **Why is it important to compare a "top match" score against a corpus-wide average baseline, rather than just trusting the ranking?**
   Because a ranking only tells you which document is *relatively* most similar among the available options - if every score in the corpus is low, the "most similar" document might still not be meaningfully related; comparing against the average baseline reveals whether a top match is a genuine standout or just the least-unrelated choice.
